# Hybrid Recommendation System — Training & Evaluation Notebook

Trains the content-based (sentence-embeddings) and collaborative (SVD)
models, evaluates before/after retraining, and does a quick manual test.

This notebook is for **experimentation only** — the production API
(`app.py`) trains and serves its own models independently with more
robust caching, staleness detection, and cold-start handling. Re-run
this notebook whenever you want to evaluate a new `content.csv` /
`interactions.csv` before deploying it.

## 1. Setup

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

# Reproducibility — the "before vs after retrain" comparison later
# generates random simulated interactions; without a fixed seed the
# comparison isn't reproducible between runs, making it hard to tell a
# real improvement from run-to-run noise.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load data

Uses a relative `data/` folder instead of a hardcoded absolute path.
The original notebook pointed at `E:\faculty\Ma'man\...`, which only
ever works on one specific machine — it silently breaks for anyone
else who clones the repo, and is exactly the kind of thing that fails
quietly in Railway/CI where that drive doesn't exist at all.

In [ ]:
# Change this if your notebook lives somewhere other than the repo root.
DATA_DIR = "data"

content_path = os.path.join(DATA_DIR, "content.csv")
interactions_path = os.path.join(DATA_DIR, "interactions.csv")
users_path = os.path.join(DATA_DIR, "users.xlsx")

print("📂 Loading data...")
content_df = pd.read_csv(content_path)
interactions_df = pd.read_csv(interactions_path)
users_df = pd.read_excel(users_path) if os.path.exists(users_path) else pd.DataFrame()

print(f"✅ Content: {content_df.shape[0]} items")
print(f"✅ Interactions: {interactions_df.shape[0]} ratings")
print(f"✅ Users: {users_df.shape[0]} users")

## 3. Clean data

Two fixes vs. the original:

1. **Index reset after `drop_duplicates()`.** `drop_duplicates()` removes
   rows but leaves gaps in the index (e.g. `[0, 1, 3, 4, 7, ...]`).
   Later, `content_sim[idx]` uses `content_df`'s index *labels* as
   *positions* into the similarity matrix. If the index isn't reset to
   a clean `0..N-1` range right after cleaning, those labels silently
   stop matching the matrix's row positions — either picking the wrong
   item's similarity row entirely, or raising an `IndexError` once a
   label exceeds the matrix size. This is the exact same class of bug
   that showed up repeatedly while debugging the API's content
   recommender.
2. **`content_id`/`user_id` cast to a consistent integer dtype.** If one
   file has them as `int64` and another as `float64`/`object` (a common
   side-effect of `NaN`s or Excel exports), `.isin()` / merges silently
   match nothing instead of raising — another bug pattern that came up
   more than once with the live API.

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean a content/interactions dataframe: drop duplicates, fill
    missing values, and reset the index so later positional lookups
    (embeddings, similarity matrices) stay aligned with row labels."""
    df = df.drop_duplicates()

    if 'title' in df.columns:
        df['title'] = df['title'].fillna('Untitled').astype(str)
    if 'description' in df.columns:
        df['description'] = df['description'].fillna('No description').astype(str)
    if 'category' in df.columns:
        df['category'] = df['category'].fillna('Uncategorized').astype(str)
    if 'level' in df.columns:
        df['level'] = df['level'].fillna('Beginner').astype(str)

    # IDs consistent as plain int64 across every file that has them —
    # prevents .isin()/merge from silently matching nothing due to a
    # dtype mismatch between content.csv and interactions.csv.
    for id_col in ('content_id', 'user_id'):
        if id_col in df.columns:
            df[id_col] = pd.to_numeric(df[id_col], errors='coerce').astype('Int64')

    # Bug fix: reset the index AFTER dropping duplicates/rows, so
    # content_df.index always matches the row positions used by
    # content_embeddings / content_sim built from it below.
    df = df.dropna(subset=[c for c in ('content_id', 'user_id') if c in df.columns])
    df = df.reset_index(drop=True)
    return df


content_df = clean_data(content_df)
interactions_df = clean_data(interactions_df)

content_df['content_id'] = content_df['content_id'].astype(int)
interactions_df['content_id'] = interactions_df['content_id'].astype(int)
interactions_df['user_id'] = interactions_df['user_id'].astype(int)

print(f"After cleaning — Content: {len(content_df)} rows, Interactions: {len(interactions_df)} rows")

## 4. Content embeddings

In [ ]:
print("🔄 Loading embedding model (first run downloads it — needs internet access once)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

content_df['text'] = (
    content_df['title'].astype(str) + " " +
    content_df['category'].astype(str) + " " +
    content_df['level'].astype(str) + " " +
    content_df['description'].astype(str)
)

content_embeddings = model.encode(content_df['text'].tolist(), show_progress_bar=True)

# content_df's index is guaranteed 0..N-1 (reset in clean_data), matching
# these rows positionally — this is what makes `content_sim[idx]` safe
# to use later.
content_sim = cosine_similarity(content_embeddings)
print(f"✅ Similarity matrix: {content_sim.shape}")

## 5. Collaborative model (SVD)

Two fixes vs. the original:

1. **Sparse matrix instead of a dense `pivot_table`.** A dense
   `user_id × content_id` matrix wastes memory proportional to
   `users × items` regardless of how few interactions actually exist —
   with real numbers like ~4,900 users × 2,000 items that's already
   ~9.8M cells for what might be a few hundred thousand actual ratings.
   `Collaborative.py` in the deployed API already uses a sparse matrix;
   this notebook now matches that so evaluation results are comparable.
2. **`n_components` capped to the data size.** `TruncatedSVD(n_components=50)`
   raises an error outright if the interaction matrix has fewer than 50
   users or items — which happens easily while testing with a small
   sample. It's now clamped to whatever the data can actually support.

In [ ]:
def retrain_collaborative(interactions_df: pd.DataFrame, n_components: int = 50, random_state: int = RANDOM_STATE):
    print("🔁 Retraining Collaborative Model...")

    user_ids = interactions_df['user_id'].unique()
    item_ids = interactions_df['content_id'].unique()
    user_map = {u: i for i, u in enumerate(user_ids)}
    item_map = {c: i for i, c in enumerate(item_ids)}

    rows = interactions_df['user_id'].map(user_map).values
    cols = interactions_df['content_id'].map(item_map).values
    data = interactions_df['rating'].values

    user_item_sparse = csr_matrix((data, (rows, cols)), shape=(len(user_ids), len(item_ids)))

    # Bug fix: n_components must stay below both matrix dimensions, or
    # TruncatedSVD raises. Small/sampled datasets hit this constantly.
    safe_n_components = max(1, min(n_components, len(item_ids) - 1, len(user_ids) - 1))
    if safe_n_components < n_components:
        print(f"⚠️ Reducing n_components {n_components} -> {safe_n_components} (not enough users/items)")

    svd = TruncatedSVD(n_components=safe_n_components, random_state=random_state)
    user_factors = svd.fit_transform(user_item_sparse)
    item_factors = svd.components_

    collab_pred = np.dot(user_factors, item_factors)

    collab_df = pd.DataFrame(collab_pred, index=user_ids, columns=item_ids)

    print("✅ Retraining Done!")
    return collab_df, user_item_sparse


collab_df, user_item = retrain_collaborative(interactions_df)

## 6. Hybrid recommendation

In [ ]:
def hybrid_recommend(user_id, collab_df, top_n=5):
    all_items = content_df['content_id'].values

    # ---------- Content Score ----------
    content_scores = pd.Series(0.0, index=all_items, dtype=float)

    history = interactions_df[interactions_df['user_id'] == user_id]['content_id'].values
    idx = content_df[content_df['content_id'].isin(history)].index

    if len(idx) > 0:
        scores = np.sum(content_sim[idx], axis=0)
        content_scores = pd.Series(scores, index=all_items)

    # ---------- Collaborative Score ----------
    if user_id in collab_df.index:
        collab_scores = collab_df.loc[user_id].reindex(all_items, fill_value=0)
    else:
        collab_scores = pd.Series(0.0, index=all_items)

    # ---------- Normalization ----------
    scaler = MinMaxScaler()
    content_norm = scaler.fit_transform(content_scores.values.reshape(-1, 1)).flatten()
    collab_norm = scaler.fit_transform(collab_scores.values.reshape(-1, 1)).flatten()

    # ---------- Final Score ----------
    final_score = 0.5 * content_norm + 0.5 * collab_norm

    result = pd.DataFrame({"content_id": all_items, "score": final_score})
    result = result.merge(content_df, on="content_id")
    result = result.sort_values("score", ascending=False)

    return result.head(top_n)

## 7. Evaluation

In [ ]:
def evaluate_model(collab_df, k=5, n_users=50):
    users = interactions_df['user_id'].unique()[:n_users]

    precisions, recalls, f1s = [], [], []

    for u in users:
        recs = hybrid_recommend(u, collab_df, top_n=k)['content_id'].tolist()

        actual = interactions_df[
            (interactions_df['user_id'] == u) & (interactions_df['rating'] >= 3)
        ]['content_id'].tolist()

        if len(actual) == 0:
            continue

        precision = len(set(recs[:k]) & set(actual)) / k
        recall = len(set(recs[:k]) & set(actual)) / len(actual)
        f1 = 0 if (precision + recall) == 0 else 2 * (precision * recall) / (precision + recall)

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    if not precisions:
        print("⚠️ No users had rating>=3 interactions to evaluate against.")
        return 0.0, 0.0, 0.0

    return np.mean(precisions), np.mean(recalls), np.mean(f1s)

In [ ]:
# BEFORE RETRAIN
collab_before = collab_df.copy()
before_metrics = evaluate_model(collab_before)
print("📊 BEFORE RETRAIN")
print("Precision:", before_metrics[0])
print("Recall:", before_metrics[1])
print("F1:", before_metrics[2])

## 8. Simulate new interactions

The simulated `content_id` range now matches the *actual* catalog size
(`content_df['content_id'].max()`) instead of a hardcoded `200` — with
the real ~2,000-item catalog, a hardcoded `200` meant every simulated
interaction only ever touched the first 10% of courses, which would
quietly bias the "after retrain" evaluation.

In [ ]:
max_content_id = int(content_df['content_id'].max())
max_user_id = int(interactions_df['user_id'].max())

n_simulated = 20000
new_data = pd.DataFrame({
    "user_id": np.random.randint(1, max_user_id + 1, n_simulated),
    "content_id": np.random.randint(1, max_content_id + 1, n_simulated),
    "rating": np.random.randint(1, 6, n_simulated)
})

interactions_updated = pd.concat([interactions_df, new_data], ignore_index=True)
print(f"Interactions: {len(interactions_df)} -> {len(interactions_updated)}")

In [ ]:
# AFTER RETRAIN
collab_after, user_item_after = retrain_collaborative(interactions_updated)
after_metrics = evaluate_model(collab_after)

print("\n📊 AFTER RETRAIN")
print("Precision:", after_metrics[0])
print("Recall:", after_metrics[1])
print("F1:", after_metrics[2])

## 9. Results comparison

In [ ]:
labels = ["Precision", "Recall", "F1"]

plt.figure(figsize=(7, 4))
plt.bar(np.arange(3) - 0.2, before_metrics, 0.4, label="Before")
plt.bar(np.arange(3) + 0.2, after_metrics, 0.4, label="After")
plt.xticks(np.arange(3), labels)
plt.title("Before vs After Retraining")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Manual test

In [ ]:
sample_user_id = int(interactions_df['user_id'].iloc[0])
print(f"Sample recommendation for user_id={sample_user_id}:\n")
hybrid_recommend(sample_user_id, collab_after)